In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import joblib
import json

df = pd.read_csv(
    "../datasets/cleaned/startup_info.csv",
    low_memory=False
)

print(df.shape)

(50000, 84)


In [2]:
scaler = MinMaxScaler()

cols = [

    "startup_health_score",
    "founder_strength_score",
    "funding_strength_score",
    "growth_score",
    "revenue",
    "investor_count",
    "valuation"

]

for col in cols:

    df[col + "_investor_norm"] = scaler.fit_transform(
        df[[col]]
    )

In [3]:
df["investor_readiness_score"] = (

      0.25 * df["startup_health_score_investor_norm"]

    + 0.20 * df["founder_strength_score_investor_norm"]

    + 0.20 * df["funding_strength_score_investor_norm"]

    + 0.15 * df["growth_score_investor_norm"]

    + 0.10 * df["valuation_investor_norm"]

    + 0.05 * df["revenue_investor_norm"]

    + 0.05 * df["investor_count_investor_norm"]

)

df["investor_readiness_score"] *= 100

In [4]:
df["investor_readiness_label"] = pd.cut(

    df["investor_readiness_score"],

    bins=[0,35,65,100],

    labels=[
        "Low",
        "Medium",
        "High"
    ]

)

print(
    df["investor_readiness_label"]
    .value_counts()
)

investor_readiness_label
Medium    40683
Low        9070
High        247
Name: count, dtype: int64


In [5]:
le = LabelEncoder()

df["investor_readiness_encoded"] = (

    le.fit_transform(
        df["investor_readiness_label"]
    )

)

print(
    dict(
        zip(
            le.classes_,
            le.transform(le.classes_)
        )
    )
)

{'High': 0, 'Low': 1, 'Medium': 2}


In [6]:
features = [

    "startup_health_score",

    "founder_strength_score",

    "funding_strength_score",

    "growth_score",

    "revenue",

    "investor_count",

    "valuation",

    "retention_rate",

    "customer_count",

    "market_size"

]

X = df[features]

y = df[
    "investor_readiness_encoded"
]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y
)

In [8]:
model = RandomForestClassifier(

    n_estimators=300,

    max_depth=12,

    random_state=42,

    n_jobs=-1
)

model.fit(
    X_train,
    y_train
)

print("Training Complete ✅")

Training Complete ✅


In [9]:
y_pred = model.predict(X_test)

In [10]:
acc = accuracy_score(
    y_test,
    y_pred
)

print(
    "Accuracy:",
    round(acc * 100,2),
    "%"
)

Accuracy: 97.8 %


In [11]:
print(

    classification_report(

        y_test,

        y_pred,

        target_names=le.classes_

    )

)

              precision    recall  f1-score   support

        High       0.84      0.78      0.81        49
         Low       0.96      0.92      0.94      1814
      Medium       0.98      0.99      0.99      8137

    accuracy                           0.98     10000
   macro avg       0.93      0.90      0.91     10000
weighted avg       0.98      0.98      0.98     10000



In [12]:
importance = pd.DataFrame({

    "Feature": features,

    "Importance": model.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

)

print(importance)

                  Feature  Importance
0    startup_health_score    0.561823
2  funding_strength_score    0.147227
1  founder_strength_score    0.101456
5          investor_count    0.080068
3            growth_score    0.057543
7          retention_rate    0.011767
6               valuation    0.010373
8          customer_count    0.009995
4                 revenue    0.009944
9             market_size    0.009805


In [14]:
joblib.dump(

    model,

    "../models/investor_model/investor_readiness.pkl"

)

joblib.dump(

    le,

    "../models/investor_model/label_encoder.pkl"

)

['../models/investor_model/label_encoder.pkl']

In [15]:
metadata = {

    "model_name": "Investor Readiness Model",

    "algorithm": "Random Forest",

    "accuracy": float(acc),

    "features": features

}

with open(

    "../models/investor_model/metadata.json",

    "w"

) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

print("Investor Model Saved ✅")

Investor Model Saved ✅


In [16]:
df.to_csv(
    "../datasets/cleaned/startup_info.csv",
    index=False
)

print("Dataset Updated ✅")

Dataset Updated ✅
